# Линейная регрессия без удаления колонок с NaN(в будущем)
- Возникла теория что у некоторых стран был NaN при регрессии в популяции и мы зная об этом исключали страну из выборки. Таких стран не было
- `year` сдвинут на 25 лет: строка с `year = t` соответствует фактическому году `t + 25`.
- Таргет `pop_plus_35` = `pop_plus_25` + 10 лет (горизонт прогноза 10 лет относительно последнего наблюдения в окне).
- Разбиение train/test делается по смещённому `year` (как в исходном ноутбуке), но список признаков формируется только по train, без использования тестовых данных.



In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# Загружаем полный набор с когортными признаками
# year сдвинут на 25 лет назад относительно фактического года наблюдения
df_all = pd.read_csv("final/filtered_25_predict10_all.csv").sort_values(["country_code", "year"])


# Разбиение как в исходном ноутбуке (смещённые годы)
train_start, train_end = 1960, 1977
test_start, test_end = 1978, 1987
horizon = test_end - test_start + 1

target_column = "pop_plus_35"
feature_cols = [c for c in df_all.columns if c not in ["country_code", target_column]]



In [7]:
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true))

def rmse_perc(y_true, y_pred):
    rmse_val = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return rmse_val / np.mean(y_true)

def sdev_perc(y_true, y_pred):
    e = np.abs(y_true - y_pred)
    return e.std() / np.mean(y_true)



In [8]:
model_name = "linear_regression_no_leak"
results = {"MAPE": [], "RMSE": [], "SDEV": []}
skipped_test_nan = 0

countries = df_all["country_code"].unique()

for c in countries:
    d = df_all[df_all["country_code"] == c]
    d_train = d[(d["year"] >= train_start) & (d["year"] <= train_end)]
    d_test  = d[(d["year"] >= test_start) & (d["year"] <= test_end)]

    # Проверяем наличие данных нужной длины
    if len(d_train) == 0 or len(d_test) != horizon:
        continue

    # Формируем признаки только по train (никакого взгляда в тест)
    usable_cols = [col for col in feature_cols if not d_train[col].isna().any()]
    if len(usable_cols) == 0:
        continue

    # Если в тесте остались NaN по этим же колонкам, просто пропускаем страну (не меняем список фич)
    if d_test[usable_cols].isna().any().any():
        skipped_test_nan += 1
        continue

    X_train = d_train[usable_cols].values
    X_test = d_test[usable_cols].values
    y_train = d_train[target_column].values
    y_test = d_test[target_column].values

    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results["MAPE"].append(mape(y_test, y_pred))
    results["RMSE"].append(rmse_perc(y_test, y_pred))
    results["SDEV"].append(sdev_perc(y_test, y_pred))

print(f"Countries processed: {len(results['MAPE'])}, skipped due to test NaN: {skipped_test_nan}")
print(f"MAPE={np.mean(results['MAPE']):.3f}, RMSE={np.mean(results['RMSE']):.3f}, SDEV={np.mean(results['SDEV']):.3f}")



Countries processed: 207, skipped due to test NaN: 0
MAPE=0.027, RMSE=0.034, SDEV=0.020
